In [ ]:
# Uncomment and run only if packages are not installed
# %pip install transformers torch numpy scikit-learn


In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity


## Model Description:
distilbert-base-uncased is a small, fast, open-source Transformer encoder model from Hugging Face.

It is a compressed version of BERT base. The word DistilBERT means it was trained using knowledge distillation, where a smaller student model learns to imitate a larger BERT teacher model.
##  Key Spec


In [2]:
import pandas as pd

specs = {
    "Feature": [
        "Model type",
        "Base model family",
        "Case handling",
        "Layers",
        "Hidden size",
        "Attention heads",
        "Parameters",
        "Vocabulary size",
        "Max input length",
        "Output",
        "Main use"
    ],
    "Value": [
        "Encoder-only Transformer",
        "BERT / DistilBERT",
        "Uncased, converts text to lowercase",
        "6 Transformer layers",
        "768",
        "12",
        "About 66 million",
        "About 30,522 tokens",
        "512 tokens",
        "Contextual embedding for each token",
        "Embeddings, classification, sentiment, NER, text understanding"
    ]
}

df = pd.DataFrame(specs)

df


,Feature,Value
0,Model type,Encoder-only Transformer
1,Base model family,BERT / DistilBERT
2,Case handling,"Uncased, converts text to lowercase"
3,Layers,6 Transformer layers
4,Hidden size,768
5,Attention heads,12
6,Parameters,About 66 million
7,Vocabulary size,"About 30,522 tokens"
8,Max input length,512 tokens
9,Output,Contextual embedding for each token


In [1]:
# Load a small open-source BERT-style encoder model
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

print("Loaded model:", model_name)


NameError: name 'AutoTokenizer' is not defined

In [ ]:
def get_model_outputs(sentence):
    encoded = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**encoded)

    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    hidden_states = outputs.last_hidden_state[0]

    return tokens, hidden_states


def get_token_embedding(sentence, target_word):
    tokens, hidden_states = get_model_outputs(sentence)

    target_tokens = tokenizer.tokenize(target_word)

    matching_positions = []

    for i in range(len(tokens) - len(target_tokens) + 1):
        if tokens[i:i + len(target_tokens)] == target_tokens:
            matching_positions.extend(range(i, i + len(target_tokens)))
            break

    if not matching_positions:
        raise ValueError(f"Could not find '{target_word}' in tokenized sentence: {tokens}")

    # If the word is split into subwords, average their vectors
    embedding = hidden_states[matching_positions].mean(dim=0).numpy()

    return embedding, tokens, matching_positions


def cos_sim(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]


In [ ]:
# Same word, different contexts
river_sentence = "The fisherman sat on the bank of the river."
finance_sentence = "She deposited money in her bank account."

bank_river_vec, river_tokens, river_pos = get_token_embedding(river_sentence, "bank")
bank_finance_vec, finance_tokens, finance_pos = get_token_embedding(finance_sentence, "bank")

print("River sentence tokens:")
print(river_tokens)
print("Position of 'bank':", river_pos)

print("\nFinance sentence tokens:")
print(finance_tokens)
print("Position of 'bank':", finance_pos)

print("\nEmbedding size:", bank_river_vec.shape)


River sentence tokens:
['[CLS]', 'the', 'fisherman', 'sat', 'on', 'the', 'bank', 'of', 'the', 'river', '.', '[SEP]']
Position of 'bank': [6]

Finance sentence tokens:
['[CLS]', 'she', 'deposited', 'money', 'in', 'her', 'bank', 'account', '.', '[SEP]']
Position of 'bank': [6]

Embedding size: (768,)


In [ ]:
# Compare the two contextual embeddings of "bank"
similarity_between_bank_meanings = cos_sim(bank_river_vec, bank_finance_vec)

print(
    "Cosine similarity between 'bank' in river context "
    f"and 'bank' in finance context: {similarity_between_bank_meanings:.4f}"
)


Cosine similarity between 'bank' in river context and 'bank' in finance context: 0.6366


# Same word token
# Different sentence context
# Different final embedding

# Cosine value	         Meaning
# 1.0	               Almost identical
# 0.8 - 0.95	       Very similar
# 0.5 - 0.75	       Related, but meaning has shifted
# 0.0	               Unrelated / orthogonal


In [7]:
# Compare "bank" with clue words from each context
river_vec, _, _ = get_token_embedding(river_sentence, "river")
money_vec, _, _ = get_token_embedding(finance_sentence, "money")
account_vec, _, _ = get_token_embedding(finance_sentence, "account")

print("Same-context comparisons:")
print("bank in river sentence vs river:", round(cos_sim(bank_river_vec, river_vec), 4))
print("bank in finance sentence vs money:", round(cos_sim(bank_finance_vec, money_vec), 4))
print("bank in finance sentence vs account:", round(cos_sim(bank_finance_vec, account_vec), 4))

print("\nCross-context comparisons:")
print("bank in river sentence vs money:", round(cos_sim(bank_river_vec, money_vec), 4))
print("bank in finance sentence vs river:", round(cos_sim(bank_finance_vec, river_vec), 4))


Same-context comparisons:
bank in river sentence vs river: 0.7927
bank in finance sentence vs money: 0.8007
bank in finance sentence vs account: 0.8378

Cross-context comparisons:
bank in river sentence vs money: 0.5419
bank in finance sentence vs river: 0.5615


In [8]:
# Sentence embeddings using mean pooling

def mean_pool_sentence_embedding(sentence):
    encoded = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**encoded)

    token_embeddings = outputs.last_hidden_state[0]
    attention_mask = encoded["attention_mask"][0]

    # Average only real tokens
    real_token_embeddings = token_embeddings[attention_mask == 1]

    return real_token_embeddings.mean(dim=0).numpy()


sentences = [
    "The fisherman sat on the bank of the river.",
    "The boat reached the shore near the river.",
    "She deposited money in her bank account.",
    "The customer opened a new savings account."
]

sentence_vectors = [mean_pool_sentence_embedding(sentence) for sentence in sentences]
sentence_similarity_matrix = cosine_similarity(sentence_vectors)

print("Sentences:")
for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence}")

print("\nSentence similarity matrix:")
print(np.round(sentence_similarity_matrix, 4))


Sentences:
0: The fisherman sat on the bank of the river.
1: The boat reached the shore near the river.
2: She deposited money in her bank account.
3: The customer opened a new savings account.

Sentence similarity matrix:
[[1.     0.8907 0.7097 0.6631]
 [0.8907 1.     0.7327 0.6963]
 [0.7097 0.7327 1.     0.8266]
 [0.6631 0.6963 0.8266 1.    ]]


the word bank does not get one fixed meaning. 
In a BERT-style encoder, its final vector changes because self-attention mixes information from surrounding words like river, money, and account.